In [105]:
import pandas as pd
from pathlib import Path
import sqlite3

con = sqlite3.connect("data/broadband.db")
%store -r threshold

RESET = False

In [106]:
if RESET:
    con.execute("DROP TABLE IF EXISTS fcc_tracts")
    con.execute("""
        CREATE TABLE fcc_tracts (
            geoid TEXT PRIMARY KEY CHECK (length(geoid) = 11),
            county TEXT NOT NULL,
            total_locations INTEGER NOT NULL CHECK (total_locations >= 0),
            no_terra_service INTEGER NOT NULL CHECK (no_terra_service BETWEEN 0 AND total_locations),
            pct_without_coverage REAL
        ) STRICT
    """)

    con.execute("DROP TABLE IF EXISTS tract_clusters")
    con.execute("""
        CREATE TABLE tract_clusters (
            geoid TEXT NOT NULL CHECK (length(geoid) = 11), 
            threshold INTEGER NOT NULL CHECK (threshold BETWEEN 0 AND 100),
            cluster INTEGER NOT NULL CHECK (cluster >= 1),
            cluster_rank INTEGER NOT NULL CHECK (cluster_rank >= 1),
            PRIMARY KEY (geoid, threshold)
        ) STRICT
    """)

    con.commit()

In [107]:
cluster_nosub_vs_noterra = pd.read_sql_query("""
    SELECT 
        SUM(s.total) AS households,
        ROUND(100.0 * (SUM(s.access_no_sub) + SUM(s.no_access) + SUM(s.sub_dialup_only)) / SUM(s.total), 2) AS pct_households_no_broadband_sub,
        c.cluster_rank AS rank,
        COUNT(*)                AS tracts,
        SUM(f.total_locations)  AS locations,
        SUM(f.no_terra_service) AS no_terra_service,
        ROUND(100.0 * SUM(f.no_terra_service) / SUM(f.total_locations), 2) AS pct_no_terra_service,
        group_concat(DISTINCT f.county) AS counties
    FROM tract_clusters AS c, acs_subs AS s
    JOIN fcc_tracts AS f ON f.geoid = c.geoid AND f.geoid = s.geoid
    WHERE c.threshold = ?
    GROUP BY c.cluster_rank
    ORDER BY c.cluster_rank
""", con, params=(threshold,))

state_nosub_vs_noterra = pd.read_sql_query("""
    SELECT 
        st.subs_total AS households,
        ROUND(100.0 * (st.subs_access_no_sub + st.subs_no_access + st.subs_sub_dialup_only) / st.subs_total, 2) AS pct_households_no_broadband_sub,
        f.locations,
        f.no_terra_service,
        ROUND(100.0 * f.no_terra_service / f.locations, 2) AS pct_no_terra_service
    FROM acs_state AS st
    CROSS JOIN (SELECT 
        SUM(total_locations) AS locations,
        SUM(no_terra_service) AS no_terra_service
        FROM fcc_tracts) AS f
""", con)

with pd.ExcelWriter("results/Clusters of Flagged Tracts.xlsx", mode="a", if_sheet_exists="replace") as writer:
    cluster_nosub_vs_noterra.to_excel(writer, sheet_name=f"Clusters {threshold}%", index=False)
    state_nosub_vs_noterra.to_excel(writer, sheet_name="North Carolina", index=False)


In [108]:
flagged_tract_demographics = pd.read_sql_query("""
    SELECT CASE WHEN c.geoid IS NULL THEN 'not flagged' ELSE 'flagged' END AS grp,
        COUNT(*) AS tracts,
        SUM(i.total) AS households,
        SUM(a.total) AS people,
        SUM(a.age_u18) AS children,
        SUM(l.lf) AS labor_force,
        ROUND(100.0 * SUM(i.inc_lt_10k + i.inc_10_20k + i.inc_20_35k) / SUM(i.total), 2) AS pct_households_earning_lt35k,
        ROUND(100.0 * SUM(age_65_plus) / SUM(a.total), 2) AS pct_aged_65_plus,
        ROUND(100.0 * SUM(age_u18_computer_no_sub + age_u18_no_computer) / SUM(a.age_u18), 2) AS pct_children_no_broadband,
        ROUND(100.0 * SUM(lf_unemployed) / SUM(l.lf), 2) AS pct_labor_froce_unemployed
    FROM fcc_tracts AS f
    LEFT JOIN tract_clusters AS c ON c.geoid = f.geoid AND c.threshold = ?
    LEFT JOIN acs_income AS i ON i.geoid = f.geoid
    LEFT JOIN acs_age AS a ON a.geoid = f.geoid
    LEFT JOIN acs_labor AS l ON l.geoid = f.geoid
    GROUP BY grp
""", con, params=(threshold,))

with pd.ExcelWriter("results/Flagged vs. Not Flagged Tracts Demographics Analysis.xlsx", mode="a", if_sheet_exists="replace") as writer:
    flagged_tract_demographics.to_excel(writer, sheet_name=f"{threshold}%", index=False)

In [109]:
cluster_demographics = pd.read_sql_query("""
    SELECT 
        c.cluster_rank AS rank,
        COUNT(*) AS tracts,
        SUM(i.total) AS households,
        SUM(a.total) AS people,
        SUM(a.age_u18) AS children,
        SUM(l.lf) AS labor_force,
        ROUND(100.0 * SUM(i.inc_lt_10k + i.inc_10_20k + i.inc_20_35k) / SUM(i.total), 2) AS pct_households_earning_lt35k,
        ROUND(100.0 * SUM(age_65_plus) / SUM(a.total), 2) AS pct_aged_65_plus,
        ROUND(100.0 * SUM(age_u18_computer_no_sub + age_u18_no_computer) / SUM(a.age_u18), 2) AS pct_children_no_broadband,
        ROUND(100.0 * SUM(lf_unemployed) / SUM(l.lf), 2) AS pct_labor_froce_unemployed
    FROM tract_clusters as c
    JOIN fcc_tracts AS f ON f.geoid = c.geoid
    LEFT JOIN acs_income AS i ON i.geoid = f.geoid
    LEFT JOIN acs_age    AS a ON a.geoid = f.geoid
    LEFT JOIN acs_labor  AS l ON l.geoid = f.geoid
    WHERE c.threshold = ?
    GROUP BY c.cluster_rank
    ORDER BY c.cluster_rank
""", con, params=(threshold,))

if Path("results/Clusters Demographics").is_file():
    with pd.ExcelWriter("results/Clusters Demographics.xlsx", mode="a", if_sheet_exists="replace") as writer:
        cluster_demographics.to_excel(writer, sheet_name=f"{threshold}%", index=False)
else:
    with pd.ExcelWriter("results/Clusters Demographics.xlsx", mode="w") as writer:
        cluster_demographics.to_excel(writer, sheet_name=f"{threshold}%", index=False)




In [110]:
#This cell was purely written by Claude 

cluster_moe_long = pd.read_sql_query("""
    WITH tr AS (
        -- one row per tract in a cluster, with every estimate + MOE the five measures need
        SELECT
            c.cluster_rank AS rank,
            s.total AS households,          s.total_moe AS households_moe,
            s.access_no_sub,                s.access_no_sub_moe,
            s.no_access,                    s.no_access_moe,
            s.sub_dialup_only,              s.sub_dialup_only_moe,
            i.total AS inc_households,      i.total_moe AS inc_households_moe,
            i.inc_lt_10k,                   i.inc_lt_10k_moe,
            i.inc_10_20k,                   i.inc_10_20k_moe,
            i.inc_20_35k,                   i.inc_20_35k_moe,
            a.total AS people,              a.total_moe AS people_moe,
            a.age_65_plus,                  a.age_65_plus_moe,
            a.age_u18,                      a.age_u18_moe,
            a.age_u18_computer_no_sub,      a.age_u18_computer_no_sub_moe,
            a.age_u18_no_computer,          a.age_u18_no_computer_moe,
            l.lf,                           l.lf_moe,
            l.lf_unemployed,                l.lf_unemployed_moe
        FROM tract_clusters AS c
        JOIN fcc_tracts AS f      ON f.geoid = c.geoid
        LEFT JOIN acs_subs AS s   ON s.geoid = f.geoid
        LEFT JOIN acs_income AS i ON i.geoid = f.geoid
        LEFT JOIN acs_age AS a    ON a.geoid = f.geoid
        LEFT JOIN acs_labor AS l  ON l.geoid = f.geoid
        WHERE c.threshold = ?
    ),
    cells (rank, measure, part, est, moe) AS (
        -- unpivot: every numerator piece is part 'x', the denominator is part 'y'
        SELECT rank, 'pct_households_no_broadband_sub', 'x', access_no_sub,           access_no_sub_moe           FROM tr UNION ALL
        SELECT rank, 'pct_households_no_broadband_sub', 'x', no_access,               no_access_moe               FROM tr UNION ALL
        SELECT rank, 'pct_households_no_broadband_sub', 'x', sub_dialup_only,         sub_dialup_only_moe         FROM tr UNION ALL
        SELECT rank, 'pct_households_no_broadband_sub', 'y', households,              households_moe              FROM tr UNION ALL

        SELECT rank, 'pct_households_earning_lt35k',    'x', inc_lt_10k,              inc_lt_10k_moe              FROM tr UNION ALL
        SELECT rank, 'pct_households_earning_lt35k',    'x', inc_10_20k,              inc_10_20k_moe              FROM tr UNION ALL
        SELECT rank, 'pct_households_earning_lt35k',    'x', inc_20_35k,              inc_20_35k_moe              FROM tr UNION ALL
        SELECT rank, 'pct_households_earning_lt35k',    'y', inc_households,          inc_households_moe          FROM tr UNION ALL

        SELECT rank, 'pct_aged_65_plus',                'x', age_65_plus,             age_65_plus_moe             FROM tr UNION ALL
        SELECT rank, 'pct_aged_65_plus',                'y', people,                  people_moe                  FROM tr UNION ALL

        SELECT rank, 'pct_children_no_broadband',       'x', age_u18_computer_no_sub, age_u18_computer_no_sub_moe FROM tr UNION ALL
        SELECT rank, 'pct_children_no_broadband',       'x', age_u18_no_computer,     age_u18_no_computer_moe     FROM tr UNION ALL
        SELECT rank, 'pct_children_no_broadband',       'y', age_u18,                 age_u18_moe                 FROM tr UNION ALL

        SELECT rank, 'pct_labor_froce_unemployed',      'x', lf_unemployed,           lf_unemployed_moe           FROM tr UNION ALL
        SELECT rank, 'pct_labor_froce_unemployed',      'y', lf,                      lf_moe                      FROM tr
    ),
    agg AS (
        -- add up estimates; MOEs combine as sqrt(sum of squares).
        -- Census rule for zero estimates: count only the single largest zero-estimate MOE, not all of them.
        SELECT
            rank, measure,
            SUM(CASE WHEN part = 'x' THEN est END) AS x,
            SUM(CASE WHEN part = 'y' THEN est END) AS y,
            COALESCE(SUM(CASE WHEN part = 'x' AND est > 0 THEN moe * moe END), 0)
              + COALESCE(MAX(CASE WHEN part = 'x' AND est = 0 THEN moe * moe END), 0) AS x_moe2,
            COALESCE(SUM(CASE WHEN part = 'y' AND est > 0 THEN moe * moe END), 0)
              + COALESCE(MAX(CASE WHEN part = 'y' AND est = 0 THEN moe * moe END), 0) AS y_moe2
        FROM cells
        GROUP BY rank, measure
    ),
    props AS (
        SELECT rank, measure, x, y, x_moe2, y_moe2,
               1.0 * x / NULLIF(y, 0) AS p              -- 1.0* avoids integer division; NULLIF avoids /0 (Durham)
        FROM agg
    ),
    moes AS (
        -- proportion MOE: (1/Y) * sqrt(MOE_X^2 - p^2 * MOE_Y^2); if that goes negative, use + instead
        SELECT rank, measure, p,
               sqrt(CASE WHEN x_moe2 - p * p * y_moe2 > 0
                         THEN x_moe2 - p * p * y_moe2
                         ELSE x_moe2 + p * p * y_moe2 END) / y AS p_moe
        FROM props
    )
    SELECT
        rank,
        measure,
        ROUND(100 * p, 2)                       AS pct,
        ROUND(100 * p_moe, 2)                   AS moe,      -- 90% confidence, in percentage points
        ROUND(100 * (p_moe / 1.645) / p, 1)     AS cv        -- coefficient of variation, %; NULL when pct is 0
    FROM moes
    ORDER BY rank, measure
""", con, params=(threshold,))

In [111]:
#This cell was purely written by Claude 

cluster_moe = cluster_moe_long.pivot(index="rank", columns="measure", values=["pct", "moe", "cv"])
cluster_moe.columns = [m if stat == "pct" else f"{m}_{stat}" for stat, m in cluster_moe.columns]
cluster_moe = cluster_moe.reset_index()

measures = ["pct_households_no_broadband_sub", "pct_households_earning_lt35k", "pct_aged_65_plus",
            "pct_children_no_broadband", "pct_labor_froce_unemployed"]
cluster_moe = cluster_moe[["rank"] + [f"{m}{s}" for m in measures for s in ("", "_moe", "_cv")]]

In [112]:
if Path("results/Clusters MOE.xlsx").is_file():
    with pd.ExcelWriter("results/Clusters MOE.xlsx", mode="a", if_sheet_exists="replace") as writer:
        cluster_moe.to_excel(writer, sheet_name=f"{threshold}%", index=False)
else:
    with pd.ExcelWriter("results/Clusters MOE.xlsx", mode="w") as writer:
        cluster_moe.to_excel(writer, sheet_name=f"{threshold}%", index=False)

In [113]:
con.close()